Lab 4
- Francisco Javier De la Torre Silva

### Import spark utils

In [11]:
from spark_utils import SparkUtils
from pyspark.sql.functions import get_json_object, col

In [5]:
spark_url = "spark://spark-master:7077"
app_name = "Lab04"
su = SparkUtils(spark_url, app_name)
su

<SparkContext master=spark://spark-master:7077 appName=Lab04>

### Generate Schemas

In [2]:
agencies_schema = SparkUtils.generate_schema([
    ("agency_id", "integer"),
    ("agency_info", "string"),
])

brands_schema = SparkUtils.generate_schema([
    ("brand_id", "integer"),
    ("brand_info", "string"),
])

cars_schema = SparkUtils.generate_schema([
    ("car_id", "integer"),
    ("car_info", "string"),
])

customers_schema = SparkUtils.generate_schema([
    ("customer_id", "integer"),
    ("customer_info", "string"),
])

rentals_schema = SparkUtils.generate_schema([
    ("rental_id", "integer"),
    ("rental_info", "string"),
])

### Load datasets

In [14]:
base_path = "../data/car_service"

agencies_df = su._spark.read.csv(f"{base_path}/agencies/agencies.csv", header=True, schema=agencies_schema)
agencies_df_json = agencies_df.select(
    col("agency_id"),
    get_json_object(col("agency_info"), "$.agency_name").alias("agency_name"),
)
brands_df = su._spark.read.csv(f"{base_path}/brands/brands.csv", header=True, schema=brands_schema)
brands_df_json = brands_df.select(
    col("brand_id"),
    get_json_object(col("brand_info"), "$.brand_name").alias("brand_name"),
)
cars_df = su._spark.read.csv(f"{base_path}/cars/cars.csv", header=True, schema=cars_schema)
cars_df_json = cars_df.select(
    col("car_id"),
    get_json_object(col("car_info"), "$.car_name").alias("car_name"),
)
customers_df = su._spark.read.csv(f"{base_path}/customers/customers.csv", header=True, schema=customers_schema)
customers_df_json = customers_df.select(
    col("customer_id"),
    get_json_object(col("customer_info"), "$.customer_name").alias("customer_name"),
)
rentals_df = su._spark.read.csv(f"{base_path}/rentals/", header=True, schema=rentals_schema)
rentals_df_json = rentals_df.select(
    col("rental_id"),
    get_json_object(col("rental_info"), "$.car_id").cast("integer").alias("car_id"),
    get_json_object(col("rental_info"), "$.customer_id").cast("integer").alias("customer_id"),
    get_json_object(col("rental_info"), "$.agency_id").cast("integer").alias("agency_id"),
)


### Generate final df

In [15]:
final_df = rentals_df_json \
    .join(cars_df_json, "car_id") \
    .join(agencies_df_json, "agency_id") \
    .join(customers_df_json, "customer_id") \
    .select("rental_id", "car_name", "agency_name", "customer_name")

In [16]:
final_df.show(5, truncate=False)

+---------+-----------------------+-------------+---------------+
|rental_id|car_name               |agency_name  |customer_name  |
+---------+-----------------------+-------------+---------------+
|11891    |Wallace-Carlson Model 9|NYC Rentals  |Margaret Jones |
|11892    |Grimes-Green Model 8   |LA Car Rental|Albert Williams|
|11893    |Stewart-Allen Model 5  |SF Cars      |Caleb Fleming  |
|11894    |Campos PLC Model 4     |NYC Rentals  |Andrew Butler  |
|11895    |Wagner LLC Model 1     |SF Cars      |Kristin Potts  |
+---------+-----------------------+-------------+---------------+
only showing top 5 rows
